# 05. CCI 최종 복합위기지수 산출

IVI, CDI, RII를 2021년 RII 기준 425개 행정동코드로 병합하고 최종 CCI를 산출합니다.

주의:
- 최종 병합 기준은 `행정동코드`입니다.
- 행정동명 단독 병합은 사용하지 않습니다.
- 자치구명 + 행정동명은 IVI 코드 부여 및 검증용으로만 사용합니다.
- 모든 지수 방향은 값이 높을수록 위험입니다.

## 1. 라이브러리 및 경로 설정

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

BASE_DIR = Path("c:/Tsum2026/T_SUM2026")
SRC_DIR = BASE_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from build_cci_final_index import (
    IVI_PATH, CDI_PATH, RII_PATH, RII_GPKG_PATH,
    OUT_FINAL_CSV, OUT_TOP_CSV, OUT_FINAL_GPKG,
    WEIGHTS, IVI_CANDIDATES,
    read_csv, clean_admin_names, normalize_code,
    prepare_ivi_to_rii_codes, validate_before_merge,
    add_rank_grade, classify_risk_type,
)

print("IVI:", IVI_PATH)
print("CDI:", CDI_PATH)
print("RII:", RII_PATH)
print("RII GPKG:", RII_GPKG_PATH)
print("가중치:", WEIGHTS)

## 2. 데이터 불러오기

In [ ]:
ivi_raw = read_csv(IVI_PATH)
cdi = read_csv(CDI_PATH)
rii = read_csv(RII_PATH)

print("IVI shape:", ivi_raw.shape)
print("CDI shape:", cdi.shape)
print("RII shape:", rii.shape)
display(ivi_raw.head())
display(cdi.head())
display(rii.head())

## 3. 컬럼명 및 행정동코드 정리

In [ ]:
# IVI 원본은 year/gu/dong 구조이며 행정동코드가 없습니다.
# RII 425개 행정동 기준의 자치구명+행정동명으로 코드를 부여한 뒤,
# 최종 병합에는 행정동코드만 사용합니다.
cdi = clean_admin_names(cdi)
rii = clean_admin_names(rii)

cdi["행정동코드"] = normalize_code(cdi["행정동코드"])
rii["행정동코드"] = normalize_code(rii["행정동코드"])

ivi = prepare_ivi_to_rii_codes(ivi_raw, rii)
ivi["행정동코드"] = normalize_code(ivi["행정동코드"])

print("IVI 후보 컬럼:", [col for col in IVI_CANDIDATES if col in ivi_raw.columns])
print("정리 후 IVI/CDI/RII shape:", ivi.shape, cdi.shape, rii.shape)
display(ivi.head())

## 4. 병합 전 검증

In [ ]:
validate_before_merge("IVI", ivi, "IVI")
validate_before_merge("CDI", cdi, "CDI")
validate_before_merge("RII", rii, "RII")

code_sets = {
    "IVI": set(ivi["행정동코드"]),
    "CDI": set(cdi["행정동코드"]),
    "RII": set(rii["행정동코드"]),
}
print("\n행정동코드 집합 일치 여부")
print("IVI == RII:", code_sets["IVI"] == code_sets["RII"])
print("CDI == RII:", code_sets["CDI"] == code_sets["RII"])
print("IVI-RII 차이:", sorted(code_sets["IVI"] ^ code_sets["RII"])[:20])
print("CDI-RII 차이:", sorted(code_sets["CDI"] ^ code_sets["RII"])[:20])

## 5. IVI + CDI + RII 병합

In [ ]:
base_cols = ["기준연도", "자치구명", "행정동명", "행정동코드", "RII", "RII_rank", "RII_grade"]
df = rii[base_cols].copy()

df = df.merge(
    ivi[["행정동코드", "IVI", "IVI_rank_recalc", "IVI_grade_recalc"]],
    on="행정동코드",
    how="left",
)
df = df.merge(
    cdi[["행정동코드", "CDI", "CDI_rank", "CDI_grade"]],
    on="행정동코드",
    how="left",
)

print("병합 후 행 수:", len(df))
print("IVI 누락 행정동")
display(df[df["IVI"].isna()][["자치구명", "행정동명", "행정동코드"]])
print("CDI 누락 행정동")
display(df[df["CDI"].isna()][["자치구명", "행정동명", "행정동코드"]])
display(df.head())

## 6. CCI 산출

In [ ]:
df["CCI"] = (
    df["IVI"] * WEIGHTS["IVI"] +
    df["CDI"] * WEIGHTS["CDI"] +
    df["RII"] * WEIGHTS["RII"]
)

print(df[["IVI", "CDI", "RII", "CCI"]].describe())
display(df.sort_values("CCI", ascending=False).head())

## 7. 순위 및 등급 산출

In [ ]:
df = add_rank_grade(df, "CCI", "CCI_rank", "CCI_grade")
display(df[["자치구명", "행정동명", "CCI", "CCI_rank", "CCI_grade"]].sort_values("CCI_rank").head(10))

## 8. 상위 30% 플래그 생성

In [ ]:
for index_col in ["IVI", "CDI", "RII"]:
    threshold = df[index_col].quantile(0.70)
    df[f"{index_col}_top30_flag"] = (df[index_col] >= threshold).astype(int)
    print(f"{index_col} top30 threshold:", threshold)

df["triple_high_flag"] = (
    (df["IVI_top30_flag"] == 1) &
    (df["CDI_top30_flag"] == 1) &
    (df["RII_top30_flag"] == 1)
).astype(int)

print("triple_high_flag 개수:", int(df["triple_high_flag"].sum()))

## 9. 위험 유형 분류

In [ ]:
df["risk_type"] = df.apply(classify_risk_type, axis=1)
print(df["risk_type"].value_counts())
display(df[["자치구명", "행정동명", "IVI_top30_flag", "CDI_top30_flag", "RII_top30_flag", "risk_type"]].head())

## 10. top_risk_dongs 생성

In [ ]:
final_cols = [
    "기준연도", "자치구명", "행정동명", "행정동코드",
    "IVI", "CDI", "RII", "CCI", "CCI_rank", "CCI_grade",
    "IVI_top30_flag", "CDI_top30_flag", "RII_top30_flag",
    "triple_high_flag", "risk_type",
]
final_df = df[final_cols].copy()
top_risk = final_df[final_df["triple_high_flag"] == 1].sort_values("CCI_rank").copy()

top_cols = [
    "자치구명", "행정동명", "행정동코드",
    "IVI", "CDI", "RII", "CCI", "CCI_rank", "CCI_grade",
    "IVI_top30_flag", "CDI_top30_flag", "RII_top30_flag",
    "triple_high_flag", "risk_type",
]
top_risk = top_risk[top_cols]

print("top risk rows:", len(top_risk))
display(top_risk.head(20))

## 11. 지도용 GPKG 생성

In [ ]:
final_gdf = None
if RII_GPKG_PATH.exists():
    import geopandas as gpd
    map_gdf = gpd.read_file(RII_GPKG_PATH)
    map_gdf["행정동코드"] = normalize_code(map_gdf["행정동코드"])
    final_gdf = map_gdf[["행정동코드", "geometry"]].merge(final_df, on="행정동코드", how="left")
    final_gdf = gpd.GeoDataFrame(final_gdf, geometry="geometry", crs=map_gdf.crs)
    print("GPKG 행 수:", len(final_gdf))
    print("GPKG CRS:", final_gdf.crs)
    display(final_gdf.head())
else:
    print("RII GPKG가 없어 지도용 파일 생성을 건너뜁니다.")

## 12. 최종 검증

In [ ]:
print("[최종 검증]")
print("최종 행 수:", len(final_df))
print("행정동코드 중복 개수:", final_df.duplicated(["행정동코드"]).sum())
print("IVI 결측치 수:", final_df["IVI"].isna().sum())
print("CDI 결측치 수:", final_df["CDI"].isna().sum())
print("RII 결측치 수:", final_df["RII"].isna().sum())
print("CCI 결측치 수:", final_df["CCI"].isna().sum())
print("CCI 범위 0~1 여부:", final_df["CCI"].between(0, 1).all())
print("CCI_grade 범위 1~5 여부:", final_df["CCI_grade"].between(1, 5).all())
print("triple_high_flag 개수:", int(final_df["triple_high_flag"].sum()))
print("risk_type별 행정동 수:")
print(final_df["risk_type"].value_counts())
print("상위 10개 CCI 행정동:")
display(
    final_df.sort_values("CCI_rank")[
        ["자치구명", "행정동명", "IVI", "CDI", "RII", "CCI", "CCI_rank", "CCI_grade", "risk_type"]
    ].head(10)
)

## 13. 파일 저장

In [ ]:
final_df.to_csv(OUT_FINAL_CSV, index=False, encoding="utf-8-sig")
top_risk.to_csv(OUT_TOP_CSV, index=False, encoding="utf-8-sig")

print("저장 완료:", OUT_FINAL_CSV)
print("저장 완료:", OUT_TOP_CSV)

if final_gdf is not None:
    final_gdf.to_file(OUT_FINAL_GPKG, layer="final_crisis_index_by_dong_2021", driver="GPKG")
    print("저장 완료:", OUT_FINAL_GPKG)